# Pipeline Orchestrator Notebook
## Purpose: Orchestrate the complete daily pipeline

**Note: Uses Spark tables (DBFS disabled in Free Edition)**

In [ ]:
# Pipeline Orchestrator
from datetime import datetime
import json

pipeline_start = datetime.now()
print("="*60)
print("FOOD DELIVERY PIPELINE - DAILY EXECUTION")
print("="*60)
print(f"Started at: {pipeline_start}")

In [ ]:
# Step 1: Data Generation
print("\n" + "-"*60)
print("STEP 1: Daily Data Generation")
print("-"*60)

try:
    # Run data generation notebook
    result = dbutils.notebook.run(
        "/Workspace/Users/YOUR_USER/databricks/01_Data_Generation_Daily.ipynb",
        1800
    )
    print("✓ Data generation complete")
    step1_status = "SUCCESS"
except Exception as e:
    print(f"✗ Error: {str(e)}")
    step1_status = "FAILED"

In [ ]:
# Step 2: Bronze Processing
print("\n" + "-"*60)
print("STEP 2: Bronze Layer Processing")
print("-"*60)

try:
    result = dbutils.notebook.run(
        "/Workspace/Users/YOUR_USER/databricks/02_Bronze_Ingestion.ipynb",
        1800
    )
    print("✓ Bronze processing complete")
    step2_status = "SUCCESS"
except Exception as e:
    print(f"✗ Error: {str(e)}")
    step2_status = "FAILED"

In [ ]:
# Step 3: Silver Transformation
print("\n" + "-"*60)
print("STEP 3: Silver Layer Transformation")
print("-"*60)

try:
    result = dbutils.notebook.run(
        "/Workspace/Users/YOUR_USER/databricks/03_Silver_Transformation.ipynb",
        1800
    )
    print("✓ Silver transformation complete")
    step3_status = "SUCCESS"
except Exception as e:
    print(f"✗ Error: {str(e)}")
    step3_status = "FAILED"

In [ ]:
# Step 4: dbt Models
print("\n" + "-"*60)
print("STEP 4: dbt Analytics Models")
print("-"*60)

try:
    result = dbutils.notebook.run(
        "/Workspace/Users/YOUR_USER/databricks/04_DBT_Models.ipynb",
        1800
    )
    print("✓ dbt models complete")
    step4_status = "SUCCESS"
except Exception as e:
    print(f"✗ Error: {str(e)}")
    step4_status = "FAILED"

In [ ]:
# Summary
pipeline_end = datetime.now()
duration = (pipeline_end - pipeline_start).total_seconds()

print("\n" + "="*60)
print("PIPELINE EXECUTION SUMMARY")
print("="*60)
print(f"Duration: {duration:.1f} seconds ({duration/60:.1f} minutes)")
print(f"\nStep Results:")
print(f"  1. Data Generation: {step1_status}")
print(f"  2. Bronze Processing: {step2_status}")
print(f"  3. Silver Transformation: {step3_status}")
print(f"  4. dbt Models: {step4_status}")

overall = "SUCCESS" if all([step1_status, step2_status, step3_status, step4_status]) == "SUCCESS" else "FAILED"
print(f"\nOverall Status: {overall}")

In [ ]:
# Return status
dbutils.notebook.exit(json.dumps({
    "status": overall,
    "duration_seconds": duration,
    "steps": {
        "data_generation": step1_status,
        "bronze": step2_status,
        "silver": step3_status,
        "dbt": step4_status
    }
}))